## Our  First Agentic Framework Project!!

Get ready for something ridiculously easy and super practical.

### What We'll Build

A simple Agent system designed to generate cold sales outreach emails, featuring:  

1. **Agent Workflow**  
   Define and orchestrate an AI agent with clear instructions and tools.

2. **Tool Integration**  
   Enable the agent to call functions or APIs seamlessly as part of its workflow.

3. **Agent Collaboration**  
   Coordinate multiple agents working together via tools and handoffs for complex tasks.

---

This project will introduce you to building real-world AI solutions with the OpenAI Agents SDK by combining instructions, tool usage, and multi-agent coordination—all without overwhelming complexity.

Stay tuned for hands-on steps and code examples coming next!

## Before We Start - SendGrid Email Setup

Please visit [SendGrid](https://sendgrid.com/) — a Twilio company specializing in scalable email delivery.

### Steps to Set Up Your SendGrid Account

1. **Sign Up for a Free Account**  
   Create your account on SendGrid (free tier available).

2. **Create an API Key**  
   - Go to **Settings** (left sidebar) → **API Keys**  
   - Click **Create API Key** (top right)  
   - Give your key a name, set permissions as needed (usually *Full Access* or *Mail Send*)  
   - Copy the generated API key to your clipboard

3. **Add API Key to Your Environment**  
   Add the following line to your `.env` file:  

   SENDGRID_API_KEY=your_api_key_here

   4. **Verify Your Sender Email**  
- Go to **Settings** → **Sender Authentication**  
- Click **Verify a Single Sender**  
- Verify your personal email address so SendGrid can send emails on your behalf

---

### Troubleshooting

If SendGrid gives you issues, there is an alternative email implementation available in  
`community_contributions/2_lab2_with_resend_email`.

---

By completing these steps, you will have an email service ready for your projects!

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [2]:
# === LLM SETUP (edit only this cell / .env to change provider/model) ===
# Defaults to Ollama.
load_dotenv(override=True)

LOCAL_LLM_API_KEY = os.getenv("LOCAL_LLM_API_KEY", "ollama")
LOCAL_LLM_BASE_URL = os.getenv("LOCAL_LLM_BASE_URL", "http://localhost:11434/v1")
LOCAL_LLM_MODEL = os.getenv("LOCAL_LLM_MODEL", "llama3.2:latest")
print(f"Using model: {LOCAL_LLM_MODEL} @ {LOCAL_LLM_BASE_URL}")

In [3]:
# Let's just check emails are working for you

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("contact@alakinfotech.com")  # Change to your verified sender
    to_email = To("reddy.revanth@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


### Did You Receive the Test Email?

If you get a **202** status code in response, then you're good to go! This means your email request was accepted and queued by SendGrid.

If not, check the following:  
- Your **API key** in the SendGrid dashboard  
- Your **verified sender email address** under **Sender Authentication**

Alternatively, you can use the email sending implementation using **"Resend Email"** located in `community_contributions/2_lab2_with_resend_email`.

Or, if you prefer, you could replace the email sending code with code to simply write the email content to a flat file for testing purposes.

## Step 1: Agent workflow

In [4]:
instructions1 = "You are a sales agent working for AlakInfoTech, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for AlakInfoTech, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for AlakInfoTech, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [5]:
local_client = AsyncOpenAI(base_url=LOCAL_LLM_BASE_URL, api_key=LOCAL_LLM_API_KEY)
local_model = OpenAIChatCompletionsModel(model=LOCAL_LLM_MODEL, openai_client=local_client)

sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=local_model
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=local_model
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=local_model
)

In [7]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Fast‑Track Your SOC 2 Readiness Without Disrupting Ops

Hi [First Name],

I know how quickly audit timelines can change – and how costly it is when the team’s focus shifts from product to paperwork.

At AlakInfoTech we’ve built an AI‑driven SaaS platform that turns your existing security processes into a live SOC 2 compliance dashboard. In under 30 days you’ll have:

* **Automated evidence collection** – no manual spreadsheets or endless emails.  
* **Real‑time risk scoring** – see where controls are strong and where gaps appear instantly.  
* **Audit‑ready reports** – PDF & JSON outputs that plug straight into the auditor’s portal.  

Our clients (e.g., [Company A], [Company B]) cut audit preparation time by 70 % and reduced their compliance budget by $25k in the first year.

Could we schedule a 15‑minute call next week to explore how this could fit with your current stack? I’ll walk you through a quick demo and share a short case study relevant to your industry.

Let me know

In [8]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Streamline Your SOC II Audit Journey with AI‑Powered Compliance

Hi [First Name],

I know how stressful the next SOC II audit can be—hours of manual documentation, constant follow‑ups, and the risk that a single oversight could delay your certification.

At **AlakInfoTech** we built an AI‑driven SaaS platform that turns that complexity into a predictable, frictionless process:

| Pain Point | How We Solve It |
|------------|-----------------|
|  Repetitive evidence collection | Automated data pulls from cloud services and on‑prem tools |
|  Unclear audit readiness | Real‑time dashboards that map controls to SOC II criteria |
|  Manual remediation tracking | AI‑assisted work‑order prioritisation & progress alerts |
|  Audit surprise findings | Continuous monitoring with instant alerting for control drift |

In the last six months, a mid‑market fintech (acquired by a Fortune‑500 bank) reduced their audit preparation time from **90 days to just 30 days** and cut remediation costs

In [10]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=local_model
)

In [11]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
**Subject:** 🚀 Turn Your SOC‑2 “Oops!” into a *Compliance* Superpower  

Hi **[First Name]**,  

Quick question: Have you ever felt like your compliance team is juggling flaming torches while riding a unicycle? 🔥🤹‍♂️  

If that sounds familiar, I’m not here to sell you another “check‑list” tool. I’m here to introduce the *one* SaaS platform that turns those flaming torches into a neatly wrapped, AI‑powered compliance bundle—so your team can actually breathe.  

### What AlakInfoTech does (in 30 seconds)  

1. **Automated Risk Scanning** – Our AI crawls every file, API call, and config change to flag potential SOC‑2 gaps *before* they become audit red‑flags.  
2. **Real‑Time Compliance Dashboard** – See your control status live; no more spreadsheet nightmares or “where did that log go?” quests.  
3. **Audit‑Ready Reports** – One click, one PDF, and you’re ready to hand over the evidence (yes, we even format it so auditors can’t get bored).  

### Why this matters for y

Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: Using Tools


In [12]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=local_model
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=local_model
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=local_model
)

In [13]:
sales_agent1

Agent(name='Professional Sales Agent', instructions='You are a sales agent working for AlakInfoTech, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoff_description=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x110993980>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Now we’re going to add a tool into the mix.

Remember all that JSON boilerplate and the `handle_tool_calls()` function with the conditional logic?  

Here’s a quick refresher:

- The agent detects when it needs to call a tool (a function or API).
- The `handle_tool_calls()` function checks which tool is requested.
- Based on the tool name, the function dispatches the corresponding handler.
- The tool executes, returning results that the agent uses to continue the conversation.

Using this pattern keeps tool integration clean, modular, and easy to extend.  

Get ready to connect your agent to real external functions!

In [14]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("contact@alakinfotech.com")  # Change to your verified sender
    to_email = To("reddy.revanth@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

### This has been automatically converted into a tool, including the necessary JSON boilerplate

In [15]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111bf4680>, strict_json_schema=True, is_enabled=True)

### You can also convert an Agent into a Tool

The OpenAI Agents SDK provides a convenient method to transform an existing Agent into a reusable Tool. This lets your agent be called by other agents or workflows as a function, enabling modular and composable AI systems.

Using the `agent.as_tool()` method, you can wrap your Agent with a tool name and description. This automatically configures the necessary interface and schema for smooth integration.

This approach is great for building networks of specialized agents that collaborate by invoking each other as tools, streamlining complex workflows with clear separation of responsibilities.

In [16]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111df8680>, strict_json_schema=True, is_enabled=True)

### Now we can gather all the tools together:

- A dedicated tool for each of our three email-writing agents
- A separate tool for the email-sending function

This modular approach makes it easy to manage and orchestrate multiple agents working in tandem, while leveraging the email-sending function as a reusable tool across workflows.

In [17]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112e1f240>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111d95120>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required'

## And now, meet our Sales Manager - the Planning Agent

This agent will serve as the central planner, orchestrating the workflow and delegating tasks to specialist agents.

It’s designed to:  
- Understand the overall sales outreach goals  
- Coordinate multiple email-writing agents  
- Manage tool usage for sending emails  
- Plan handoffs and collaboration between agents  

Our Sales Manager brings structure and strategic oversight, ensuring smooth operation and maximized impact in the sales outreach process.

In [19]:
instructions ="You are a sales manager working for ComplAI. You use the tools given to you to generate cold sales emails. \
You never generate sales emails yourself; you always use the tools. \
You try all 3 sales_agent tools once before choosing the best one. \
You pick the single best email and use the send_email tool to send the best email (and only the best email) to the user."


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=local_model)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

### Windows 11 SSL Certificate Fix (Thanks to Oleksandr!)

If you're experiencing SSL certificate errors when sending emails, here's a quick fix that works well on Windows 11:

**Step 1: Update the certifi package**

**Step 2: Set the SSL certificate path in your code**

This solution helps Python locate and use the correct SSL certificates for secure email connections. The `certifi` package contains Mozilla's trusted Certificate Authority bundle, which resolves most SSL verification issues.

**Alternative approaches if the above doesn't work:**
- Try `pip install --upgrade certifi` if using regular pip instead of uv
- For persistent issues, you can also set the environment variable system-wide

**A big thank you to student Oleksandr for sharing this Windows 11 solution!** 🎉


## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

### Agent Handoffs vs. Agents-as-Tools

**Handoffs** let one agent delegate control directly to another:

- Agent A decides it’s appropriate to hand off a subtask  
- It transfers *all* context and control to Agent B  
- Agent B continues the conversation or workflow  
- Control does *not* return automatically; Agent B drives the next steps

**Agents-as-Tools** let agents collaborate via function-like calls:

- Agent A invokes Agent B as a “tool”  
- Agent B runs, produces a result, and returns control to Agent A  
- Agent A resumes, using B’s output in its ongoing workflow  

| Feature               | Handoffs                              | Agents-as-Tools                       |
|-----------------------|---------------------------------------|---------------------------------------|
| Control flow          | Passes *across* to another agent      | Passes *back* after tool invocation   |
| Context transfer      | Full context handed off               | Only the specified input/parameters   |
| Return behavior       | No automatic return; B continues work | Returns to A automatically with result|
| Use case              | Long-running collaborations           | Short, function-like interactions     |


In [20]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=local_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=local_model )
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("contact@alakinfotech.com")  # Change to your verified sender
    to_email = To("reddy.revanth@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [22]:
tools = [subject_tool, html_tool, send_html_email]

In [23]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111d95580>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111d953a0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='send_html_email', description='Send out an email with the given subject and HTML body to all sales pros

In [24]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model=local_model,
    handoff_description="Convert an email to HTML and send it")

### Now we have 3 tools and 1 handoff

In [25]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112e1f240>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x111d95120>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': 

In [26]:
sales_manager_instructions = "You are a sales manager working for ComplAI. You use the tools given to you to generate cold sales emails. \
You never generate sales emails yourself; you always use the tools. \
You try all 3 sales agent tools at least once before choosing the best one. \
You can use the tools multiple times if you're not satisfied with the results from the first try. \
You select the single best email using your own judgement of which email will be most effective. \
After picking the email, you handoff to the Email Manager agent to format and send the email."


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=local_model)

message = "Send out a cold sales email addressed to Dear CEO from Revanth"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)